# 05 · A abordagem quântica da DualQ, aplicada passo a passo

**Posição deste notebook na sequência:**

- `01_fundamento_teorico.ipynb`: teoria e baseline clássico (Markowitz) com os 4 ativos de 2023–2025.
- `02` a `04`: nossa abordagem (QUBO, QAOA, alocação com pesos e VQE).
- **`05` (este):** a abordagem quântica da DualQ, com os mesmos hiperparâmetros dela, para comparar com a nossa.

**O que fazemos aqui, em três passos:**

1. **Reproduzir o caso da DualQ**, com os dados e as unidades dela (2016–2021, BRKM5/ITUB4/VALE3/KLBN4). Se nossa reimplementação der o mesmo ótimo e valor que o notebook dela, ela está validada.
2. **Aplicar a mesma abordagem aos nossos dados** (01, 2023–2024 de treino), mantendo os hiperparâmetros da DualQ.
3. **Isolar o efeito das unidades:** a DualQ mistura retorno anual com covariância diária. Testamos o mesmo problema com as duas unidades consistentes.

**Hiperparâmetros da DualQ (de cada célula do notebook dela):**

| Item | Valor | Célula |
|---|---|---|
| Apetite ao risco $q$ | 0,5 | 61 |
| Orçamento $B$ | `num_assets // 2` = 2 | 61 |
| Penalidade | `penalty = num_assets` = 4 | 61 |
| QAOA | $p = 20$, COBYLA `maxiter=250`, seed 3 | 68 |
| VQE | `TwoLocal(…, 'ry', 'cz', reps=11, entanglement='full')`, COBYLA `maxiter=500`, seed 215 | 71 |

**Resultados que a DualQ mostra nas saídas salvas do notebook dela** (lidos das células de saída, não reexecutados por nós):

| Método | Ótimo encontrado | Valor | Probabilidade do ótimo |
|---|---|---|---|
| Solução exata (célula 66) | [1 0 0 1] | −0,6152 | 1,0000 |
| QAOA, $p = 20$ (célula 68) | [1 0 0 1] | −0,6152 | 0,6329 |
| VQE, 11 repetições (célula 71) | [1 0 0 1] | −0,6152 | 0,4897 |

Esses números são o que precisamos reproduzir. Note que a DualQ reporta **probabilidade alta** no ótimo, bem diferente do que obtivemos em 03 e 04 (probabilidade de poucos por cento). A seção 5 mostra por que.

In [4]:
import warnings
warnings.filterwarnings('ignore', message='.*SparseEfficiency.*')
warnings.filterwarnings('ignore', category=Warning, module='scipy.sparse')
warnings.filterwarnings('ignore', category=DeprecationWarning)

import os
import numpy as np
import pandas as pd
from itertools import product

from qiskit_optimization import QuadraticProgram
from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit_algorithms.minimum_eigensolvers import QAOA, SamplingVQE
from qiskit_algorithms.optimizers import COBYLA
from qiskit.primitives import StatevectorSampler
from qiskit.circuit.library import n_local

np.set_printoptions(precision=4, suppress=True)

# Hiperparâmetros da DualQ (células 61, 68 e 71)
Q_RISCO = 0.5
B_ORC = 2
PENALIDADE = 4.0
SHOTS = 8192

def qubo_selecao(mu_v, Sig, q=Q_RISCO, B=B_ORC, lam=PENALIDADE):
    """f(x) = q xᵀΣx − μᵀx + λ(1ᵀx − B)², escrito como xᵀQx + cᵀx + d (mesma forma da DualQ e do notebook 02)."""
    n_ = len(mu_v)
    um = np.ones(n_)
    Q = q * Sig + lam * np.outer(um, um)
    c = -mu_v - 2 * lam * B * um
    d = lam * B ** 2
    return Q, c, d

def qp_selecao(mu_v, Sig, nomes):
    Q, c, d = qubo_selecao(mu_v, Sig)
    qp = QuadraticProgram('dualq')
    for a in nomes:
        qp.binary_var(name=a)
    qp.minimize(constant=d, linear=c, quadratic=Q)
    return qp, Q, c, d

def rodar_dualq(mu_v, Sig, nomes, rotulo):
    """Exato + QAOA (p=20) + VQE (reps=11), com os hiperparâmetros da DualQ."""
    n_ = len(nomes)
    qp, Q, c, d = qp_selecao(mu_v, Sig, nomes)
    estados = np.array(list(product([0, 1], repeat=n_)))
    energia = np.einsum('ij,jk,ik->i', estados, Q, estados) + estados @ c + d
    viavel = estados.sum(axis=1) == B_ORC
    idx_v = np.where(viavel)[0]
    k_ot = idx_v[np.argmin(energia[idx_v])]
    otimos = [estados[i] for i in idx_v if abs(energia[i] - energia[k_ot]) < 1e-9]
    ot = estados[k_ot]

    def metricas(res):
        p_opt = sum(s.probability for s in res.samples if any(np.array_equal(s.x, o) for o in otimos))
        p_viav = sum(s.probability for s in res.samples if s.x.sum() == B_ORC)
        return p_opt, p_viav

    # QAOA p=20 (célula 68): COBYLA 250 iterações, seed 3
    np.random.seed(3)
    qaoa = QAOA(sampler=StatevectorSampler(default_shots=SHOTS, seed=3),
                optimizer=COBYLA(maxiter=250), reps=20,
                initial_point=np.random.default_rng(3).uniform(-np.pi, np.pi, 40))
    r_qaoa = MinimumEigenOptimizer(qaoa).solve(qp)

    # VQE 11 repetições (célula 71): COBYLA 500 iterações, seed 215
    ansatz = n_local(n_, ['ry'], ['cz'], entanglement='full', reps=11)
    vqe = SamplingVQE(sampler=StatevectorSampler(default_shots=SHOTS, seed=215), ansatz=ansatz,
                      optimizer=COBYLA(maxiter=500),
                      initial_point=np.random.default_rng(215).uniform(-np.pi, np.pi, ansatz.num_parameters))
    r_vqe = MinimumEigenOptimizer(vqe).solve(qp)

    po_q, pv_q = metricas(r_qaoa)
    po_v, pv_v = metricas(r_vqe)
    return {
        'caso': rotulo,
        'ótimo exato': ''.join(map(str, ot)),
        'valor exato': round(energia[k_ot], 4),
        'QAOA x': ''.join(str(int(v)) for v in r_qaoa.x),
        'QAOA P(ótimo)': round(po_q, 4),
        'QAOA P(viável)': round(pv_q, 4),
        'VQE x': ''.join(str(int(v)) for v in r_vqe.x),
        'VQE P(ótimo)': round(po_v, 4),
        'VQE P(viável)': round(pv_v, 4),
        'otimos empatados': len(otimos),
    }

print('Funções prontas. Hiperparâmetros: q =', Q_RISCO, '| B =', B_ORC, '| λ =', PENALIDADE, '| shots =', SHOTS)

Funções prontas. Hiperparâmetros: q = 0.5 | B = 2 | λ = 4.0 | shots = 8192


## 1. Reproduzindo o caso da DualQ

Usamos os mesmos ativos (`BRKM5`, `ITUB4`, `VALE3`, `KLBN4`) e o mesmo período (2016-01-01 a 2021-09-20). Como o notebook original calcula:

- $\mu$ = média dos retornos **anuais** (`resample('Y').last().pct_change().mean()`, célula 39);
- $\Sigma$ = covariância dos retornos **diários** (`pct_change().cov()`, célula 30). **Essa é a mistura de unidades** que apontamos em `audit_quantum.md` (A-01).

Baixamos os dados com `yfinance`. Se o download falhar, o notebook para aqui e registramos isso.

In [5]:
import yfinance as yf

ativos_dualq = ['BRKM5.SA', 'ITUB4.SA', 'VALE3.SA', 'KLBN4.SA']
dados_dq = yf.download(ativos_dualq, start='2016-01-01', end='2021-09-20',
                       auto_adjust=False, progress=False)['Close']
dados_dq = dados_dq[ativos_dualq].dropna()
print('Período:', dados_dq.index.min().date(), 'a', dados_dq.index.max().date(), '| observações:', len(dados_dq))

mu_dq = (dados_dq.resample('YE').last().pct_change().mean()).values
Sig_dq_diaria = dados_dq.pct_change().cov().values
print('\nμ (anual, como na DualQ):', dict(zip(ativos_dualq, mu_dq.round(4))))
print('Variância diária média:', round(float(np.mean(np.diag(Sig_dq_diaria))), 6))

Período: 2016-01-04 a 2021-09-17 | observações: 1422

μ (anual, como na DualQ): {'BRKM5.SA': np.float64(0.3154), 'ITUB4.SA': np.float64(0.0571), 'VALE3.SA': np.float64(0.3011), 'KLBN4.SA': np.float64(0.1143)}
Variância diária média: 0.000702


In [6]:
# Verificação: o valor de f(x) para [1 0 0 1] deve ser o da célula 66 da DualQ (−0,6152)
Q_dq, c_dq, d_dq = qubo_selecao(mu_dq, Sig_dq_diaria)
x_dq = np.array([1, 0, 0, 1])
valor_dq = x_dq @ Q_dq @ x_dq + c_dq @ x_dq + d_dq
print(f'f([1 0 0 1]) com os dados reproduzidos = {valor_dq:.4f}  (DualQ, célula 66: −0,6152)')

f([1 0 0 1]) com os dados reproduzidos = -0.4288  (DualQ, célula 66: −0,6152)


Se o valor bater, a formulação está igual à da DualQ. Agora rodamos o caso completo: exato, QAOA e VQE, com os mesmos hiperparâmetros.

In [4]:
linha_reproducao = rodar_dualq(mu_dq, Sig_dq_diaria, ativos_dualq,
                              'Reprodução DualQ (dados e unidades da DualQ)')
pd.DataFrame([linha_reproducao]).T

,0
caso,Reprodução DualQ (dados e unidades da DualQ)
ótimo exato,1010
valor exato,-0.6152
QAOA x,1010
QAOA P(ótimo),0.0122
QAOA P(viável),0.1401
VQE x,1010
VQE P(ótimo),0.0668
VQE P(viável),0.5403
otimos empatados,1


**Leitura:** se o ótimo exato for `1001` com valor −0,6152, nossa formulação coincide com a da DualQ. Os valores de probabilidade de QAOA e VQE mostram se nossa execução também concentra o resultado no ótimo, como a DualQ reporta (0,63 e 0,49). Diferenças de probabilidade podem vir de dois lugares: a semente e o ponto inicial, que a DualQ não fixou para o VQE, e a versão do Qiskit, já que ela usou `qiskit.aqua`.

**Verificação 1 · a ordem dos ativos.** A DualQ rotula o vetor `[1 0 0 1]` como BRKM5 + KLBN4. Nossa reprodução dá o mesmo valor (−0,6152) para `1010`, ou seja, BRKM5 + VALE3. Como o valor é idêntico, o problema é o mesmo, e a diferença está no rótulo: na ordem alfabética dos dados (`BRKM5, ITUB4, KLBN4, VALE3`), a posição 3 é KLBN4; na ordem da lista da DualQ (`BRKM5, ITUB4, VALE3, KLBN4`), a posição 3 é VALE3. Se a DualQ montou o vetor com uma ordem e rotulou com outra, o rótulo do ótimo dela está trocado. Isso é o mesmo tipo de erro do E-08 (`errors.md`).

**Verificação 2 · as sementes.** Uma execução pode ter tido sorte ou azar. Repetimos o caso de reprodução com 8 sementes para o QAOA e para o VQE, e olhamos a distribuição de `P(ótimo)`.

In [7]:
def sementes_dualq(mu_v, Sig, nomes, seeds):
    qp, Q, c, d = qp_selecao(mu_v, Sig, nomes)
    n_ = len(nomes)
    estados = np.array(list(product([0, 1], repeat=n_)))
    energia = np.einsum('ij,jk,ik->i', estados, Q, estados) + estados @ c + d
    viavel = estados.sum(axis=1) == B_ORC
    idx_v = np.where(viavel)[0]
    k_ot = idx_v[np.argmin(energia[idx_v])]
    ot_set = [estados[i] for i in idx_v if abs(energia[i] - energia[k_ot]) < 1e-9]
    linhas = []
    for s in seeds:
        qaoa = QAOA(sampler=StatevectorSampler(default_shots=SHOTS, seed=s), optimizer=COBYLA(maxiter=250),
                    reps=20, initial_point=np.random.default_rng(s).uniform(-np.pi, np.pi, 40))
        r = MinimumEigenOptimizer(qaoa).solve(qp)
        po = sum(x.probability for x in r.samples if any(np.array_equal(x.x, o) for o in ot_set))
        ansatz = n_local(n_, ['ry'], ['cz'], entanglement='full', reps=11)
        vqe = SamplingVQE(sampler=StatevectorSampler(default_shots=SHOTS, seed=s), ansatz=ansatz,
                          optimizer=COBYLA(maxiter=500),
                          initial_point=np.random.default_rng(s).uniform(-np.pi, np.pi, ansatz.num_parameters))
        rv = MinimumEigenOptimizer(vqe).solve(qp)
        pv = sum(x.probability for x in rv.samples if any(np.array_equal(x.x, o) for o in ot_set))
        linhas.append({'semente': s, 'QAOA P(ótimo)': round(po, 4), 'VQE P(ótimo)': round(pv, 4)})
    return pd.DataFrame(linhas)

sem_dq = sementes_dualq(mu_dq, Sig_dq_diaria, ativos_dualq, range(8))
print(sem_dq.to_string(index=False))
print('\nDualQ (saídas salvas): QAOA 0,6329 | VQE 0,4897')
print(sem_dq[['QAOA P(ótimo)', 'VQE P(ótimo)']].describe().round(4).to_string())

 semente  QAOA P(ótimo)  VQE P(ótimo)
       0         0.0461        0.0001
       1         0.0065        0.0000
       2         0.0023        0.4376
       3         0.0122        0.0211
       4         0.0980        0.7272
       5         0.0079        0.0200
       6         0.0082        0.0025
       7         0.1807        0.0147

DualQ (saídas salvas): QAOA 0,6329 | VQE 0,4897
       QAOA P(ótimo)  VQE P(ótimo)
count         8.0000        8.0000
mean          0.0452        0.1529
std           0.0636        0.2763
min           0.0023        0.0000
25%           0.0076        0.0019
50%           0.0102        0.0174
75%           0.0591        0.1252
max           0.1807        0.7272


## 2. A mesma abordagem nos nossos dados (01)

Agora trocamos apenas os **dados**: os 4 ativos da 01 (BBAS3, ITUB4, KLBN4, VALE3), treino 2023–2024. Mantemos $q$, $B$, $\lambda$ e os hiperparâmetros da DualQ.

Testamos duas unidades:

- **Unidades da DualQ:** $\mu$ anual, $\Sigma$ diária. A mesma mistura do caso anterior.
- **Unidades consistentes:** $\mu$ e $\Sigma$ ambos anuais, como fizemos nos notebooks 02 a 04.

In [6]:
df01 = pd.read_csv('../data/precos_b3.csv', header=[0, 1], index_col=0, skiprows=[2], parse_dates=True)
close01 = df01['Close'].apply(pd.to_numeric, errors='coerce').dropna()
close01.index = pd.to_datetime(close01.index)
ativos01 = list(close01.columns)

ret01 = close01.loc['2023-01-01':'2024-12-31'].pct_change().dropna()
mu01_anual = (ret01.mean() * 252).values
Sig01_diaria = ret01.cov().values
Sig01_anual = Sig01_diaria * 252

print('Ativos (01):', ativos01)
print('μ anual:', dict(zip(ativos01, mu01_anual.round(4))))

Ativos (01): ['BBAS3.SA', 'ITUB4.SA', 'KLBN4.SA', 'VALE3.SA']
μ anual: {'BBAS3.SA': np.float64(0.2946), 'ITUB4.SA': np.float64(0.1956), 'KLBN4.SA': np.float64(0.2037), 'VALE3.SA': np.float64(-0.1331)}


In [7]:
linha_unid_dq = rodar_dualq(mu01_anual, Sig01_diaria, ativos01,
                            '01 · unidades da DualQ (μ anual, Σ diária)')
linha_unid_cons = rodar_dualq(mu01_anual, Sig01_anual, ativos01,
                              '01 · unidades consistentes (μ e Σ anuais)')
resultados = pd.DataFrame([linha_reproducao, linha_unid_dq, linha_unid_cons]).set_index('caso')
resultados.T

caso,Reprodução DualQ (dados e unidades da DualQ),"01 · unidades da DualQ (μ anual, Σ diária)",01 · unidades consistentes (μ e Σ anuais)
ótimo exato,1010,1010,1010
valor exato,-0.6152,-0.4981,-0.4465
QAOA x,1010,1010,1010
QAOA P(ótimo),0.0122,0.0975,0.0166
QAOA P(viável),0.1401,0.2267,0.1936
VQE x,1010,1010,1010
VQE P(ótimo),0.0668,0.7396,0.6373
VQE P(viável),0.5403,0.7461,0.6549
otimos empatados,1,1,1


## 3. Comparação com o que a DualQ mostra

Compare as linhas com a tabela da introdução:

- A **probabilidade alta** de ótimo da DualQ aparece quando as unidades não são consistentes? Ou ela depende do conjunto de dados?
- Com unidades consistentes, o problema muda de caráter? A penalidade $\lambda = 4$ passa a dominar o termo de risco, como vimos em 02?
- Os nossos resultados em 03 e 04 usaram $\lambda$ diferente e $p$ diferente. Esta comparação isola o efeito de cada escolha.

## 4. Uma leitura cuidadosa

Antes de concluir algo, lembre que:

- Cada linha é **uma execução**, com uma semente. Para comparar probabilidades, seria preciso repetir com várias sementes.
- A DualQ não mostra o número de sementes nem a variância. Sem isso, não sabemos quão estável é o resultado dela.
- Dois anos de treino e um de teste não bastam para julgar a qualidade da carteira. O foco aqui é o **comportamento do algoritmo**, e não a escolha de investimento.

**Verificação 3 · sementes nos dados de 01.** Os números de uma única semente acima não são confiáveis. Repetimos a varredura de 8 sementes para as duas unidades nos dados de 01.

In [8]:
sem_unid_dq = sementes_dualq(mu01_anual, Sig01_diaria, ativos01, range(8))
sem_unid_cons = sementes_dualq(mu01_anual, Sig01_anual, ativos01, range(8))
resumo_sem = pd.DataFrame({
    'unidades DualQ · QAOA (média / mín / máx)': [sem_unid_dq['QAOA P(ótimo)'].mean(), sem_unid_dq['QAOA P(ótimo)'].min(), sem_unid_dq['QAOA P(ótimo)'].max()],
    'unidades DualQ · VQE (média / mín / máx)': [sem_unid_dq['VQE P(ótimo)'].mean(), sem_unid_dq['VQE P(ótimo)'].min(), sem_unid_dq['VQE P(ótimo)'].max()],
    'consistentes · QAOA (média / mín / máx)': [sem_unid_cons['QAOA P(ótimo)'].mean(), sem_unid_cons['QAOA P(ótimo)'].min(), sem_unid_cons['QAOA P(ótimo)'].max()],
    'consistentes · VQE (média / mín / máx)': [sem_unid_cons['VQE P(ótimo)'].mean(), sem_unid_cons['VQE P(ótimo)'].min(), sem_unid_cons['VQE P(ótimo)'].max()],
}, index=['média', 'mínimo', 'máximo']).T.round(4)
resumo_sem

,média,mínimo,máximo
unidades DualQ · QAOA (média / mín / máx),0.0578,0.0033,0.0975
unidades DualQ · VQE (média / mín / máx),0.1484,0.0000,0.4989
consistentes · QAOA (média / mín / máx),0.0288,0.0002,0.1104
consistentes · VQE (média / mín / máx),0.1314,0.0000,0.4690


**Leitura final desta seção:** a DualQ reporta probabilidade alta de ótimo (0,63 QAOA, 0,49 VQE) com **uma** execução. Nossa varredura de sementes mostra que esse número depende muito da semente. Compare a média e a faixa acima, não um único valor.